In [6]:
import heapq
import math

class Node:
    """
    Represents a discrete search state in grid space.
    Priority in the min-heap is governed by total evaluation cost f = g + h.
    Tie-breaking prioritizes smaller heuristic distance h to drive exploration toward the goal.
    """
    def __init__(self, position, parent=None, g=0.0, h=0.0):
        self.position = position  # Coordinate tuple: (row, col)
        self.parent = parent      # Reference to predecessor Node instance
        self.g = g                # Cumulative path cost from start to current node
        self.h = h                # Heuristic estimate from current node to goal
        self.f = g + h            # Evaluation function priority

    def __lt__(self, other):
        if self.f == other.f:
            return self.h < other.h
        return self.f < other.f

    def __eq__(self, other):
        return self.position == other.position

    def __hash__(self):
        return hash(self.position)

def manhattan_distance(p1, p2):
    """Calculates L1 norm distance between two 2D grid coordinates."""
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

def solve_aliceland_rover(grid, start, goal):
    """
    Finds the minimum-cost traversal path for the Aliceland Relief Rover.

    Parameters:
        grid (list of list of int): Grid matrix with costs (1, 3, 5) or wall (-1).
        start (tuple): Starting coordinate (row, col).
        goal (tuple): Target coordinate (row, col).

    Returns:
        tuple: (optimal_path_list, minimum_total_cost)
    """
    rows = len(grid)
    cols = len(grid[0])

    start_node = Node(start, None, g=0.0, h=manhattan_distance(start, goal))
    open_heap = []
    heapq.heappush(open_heap, start_node)

    open_dict = {start: start_node}
    closed_set = set()

    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    while open_heap:
        current_node = heapq.heappop(open_heap)

        if current_node.position in closed_set:
            continue

        closed_set.add(current_node.position)

        if current_node.position == goal:
            path = []
            curr = current_node
            while curr:
                path.append(curr.position)
                curr = curr.parent
            return path[::-1], current_node.g

        r, c = current_node.position

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            neighbor_pos = (nr, nc)

            if 0 <= nr < rows and 0 <= nc < cols:
                # TODO 1: Skip neighbor if it is an impassable floodwall (-1)
                #         or if it has already been closed.
                if grid[nr][nc] == -1 or neighbor_pos in closed_set:
                  continue

                # TODO 2: Calculate step cost based on target cell terrain value
                #         and compute tentative_g cost.

                step_cost = float(grid[nr][nc])
                tentative_g = current_node.g + step_cost

                if neighbor_pos in open_dict and tentative_g >= open_dict[neighbor_pos].g:
                    continue

                neighbor_node = Node(
                    position=neighbor_pos,
                    parent=current_node,
                    g=tentative_g,
                    h=manhattan_distance(neighbor_pos, goal)
                )

                open_dict[neighbor_pos] = neighbor_node
                heapq.heappush(open_heap, neighbor_node)

    return None, float('inf')


# Grid Encoding: -1 = Floodwall, 1 = Paved Road, 3 = Muddy Track, 5 = Flooded Marsh
aliceland_grid = [
    [1,  1,  5,  1],
    [-1, 3,  1,  1],
    [1,  1,  5,  1],
    [1, -1,  1,  1]
]

start_coord = (0, 0)
goal_coord = (3, 3)

path, cost = solve_aliceland_rover(aliceland_grid, start_coord, goal_coord)
print("Discovered Path:", path)
print("Total Traversal Cost:", cost)

Discovered Path: [(0, 0), (0, 1), (1, 1), (1, 2), (1, 3), (2, 3), (3, 3)]
Total Traversal Cost: 8.0
